In [ ]:
!pip -q install sqlalchemy psycopg2-binary joblib lightgbm

In [ ]:
# =========================
# Imports / Drive Mount
# =========================
import io
import shap
import os
import re
import json
import joblib
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sqlalchemy import create_engine, text

warnings.filterwarnings("ignore")


In [ ]:
# =========================
# Config
# =========================

# 1) DB 접속 정보
# - GitHub Actions에서는 Repository Secrets의 PG_URL 또는 DATABASE_URL을 사용
# - 로컬/코랩에서만 직접 문자열로 넣어 테스트 가능
PG_URL = os.getenv("PG_URL") or os.getenv("DATABASE_URL")
if PG_URL is None or str(PG_URL).strip() == "":
    raise ValueError("PG_URL 또는 DATABASE_URL 환경변수가 비어 있습니다.")

# 2) DB 테이블
MODEL_REGISTRY_TABLE = "model_registry"
MODEL_FEATURES_TABLE = "model_features"
PREDICTIONS_TABLE = "predictions"

# model_features에서 사용할 model_id
FEATURE_MODEL_ID = "model1"

# 3) 예측 설정
AREA_NAME = "육지"
PRED_MODEL_ID = "smp_combined"

# 직접 지정하려면 예: "2026-07-07 00:00:00"
# None이면 PRED_START_MODE에 따라 자동 결정
PRED_START = os.getenv("PRED_START") or None

# 운영 기본값:
# - tomorrow_kst: 실행일(KST) 다음날 00시부터 예측
#   예) 2026-07-06 실행 → 2026-07-07 00:00부터 예측
# - today_kst: 실행일(KST) 00시부터 예측
# - after_latest_actual_hour: model_features의 마지막 실제 SMP 다음 1시간부터 예측
# - after_latest_actual_day: 기존 방식. 마지막 실제 SMP 날짜의 다음날 00시부터 예측
PRED_START_MODE = os.getenv("PRED_START_MODE", "tomorrow_kst")

# 테스트/재현용 실행 기준일. 예: "2026-07-06"
# 비워두면 실제 현재 KST 날짜를 사용
PRED_RUN_DATE_KST = os.getenv("PRED_RUN_DATE_KST") or None

# 일주일치 = 168시간
HORIZON_HOURS = int(os.getenv("HORIZON_HOURS", "168"))

# 현재 요청 기준: lag1까지 recursive 사용
USE_RECURSIVE_SMP_LAG1 = True

# active model을 불러온다. 특정 버전 고정하려면 예: "v1.0"
MODEL_VERSION = os.getenv("MODEL_VERSION") or None

# 저장 방식: 기존 같은 기간/area/model_id 예측 삭제 후 append
SAVE_MODE = "delete_insert"

# 결과 확인용
DISPLAY_N = 30


In [ ]:
# =========================
# Basic Utilities
# =========================

DATETIME_COL = "datetime"
TARGET_COL = "smp"

def get_engine():
    if PG_URL is None or str(PG_URL).strip() == "":
        raise ValueError("PG_URL이 비어 있습니다.")

    engine = create_engine(
        PG_URL,
        pool_pre_ping=True
    )
    return engine


def test_db_connection(engine):
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1")).scalar()
    print("DB connection test:", result)


def to_naive_datetime_series(s):
    s = pd.to_datetime(s, errors="coerce", utc=True)
    return s.dt.tz_convert(None)


def to_naive_timestamp(x):
    ts = pd.to_datetime(x, errors="coerce")
    if pd.isna(ts):
        return pd.NaT
    if ts.tzinfo is not None:
        ts = ts.tz_convert(None)
    return pd.Timestamp(ts).floor("h")


def to_db_value(x):
    if x is None:
        return None
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, np.floating):
        return float(x)
    if isinstance(x, np.bool_):
        return bool(x)
    if isinstance(x, pd.Timestamp):
        return x.to_pydatetime()
    return x


def percentile_score_from_train(reference_values, values):
    ref = pd.Series(reference_values).dropna().astype(float).values
    val = pd.Series(values).astype(float).values

    if len(ref) == 0:
        return np.zeros(len(val))

    ref_sorted = np.sort(ref)
    scores = (
        np.searchsorted(ref_sorted, val, side="right")
        / len(ref_sorted)
        * 100
    )

    return np.clip(scores, 0, 100)

def percentile_rank_from_reference(reference_values, values):
    ref = pd.Series(reference_values).dropna().astype(float).values
    val = pd.Series(values).astype(float).values

    if len(ref) == 0:
        return np.zeros(len(val))

    ref_sorted = np.sort(ref)
    rank = np.searchsorted(ref_sorted, val, side="right") / len(ref_sorted)

    return np.clip(rank, 0, 1)


def compute_pssr_hybrid_weight_for_prediction(
    q75_pred,
    train_q75_pred,
    dr_score_values=None,
    config=None,
):
    if config is None:
        config = {
            "score_center": 60.0,
            "score_range": 30.0,
            "rank_start": 0.70,
            "rank_range": 0.30,
            "score_weight": 0.5,
            "rank_weight": 0.5,
        }

    q75_pred = np.asarray(q75_pred, dtype=float)

    rank_weight = percentile_rank_from_reference(
        train_q75_pred,
        q75_pred
    )

    rank_weight = (
        (rank_weight - config["rank_start"])
        / config["rank_range"]
    )
    rank_weight = np.clip(rank_weight, 0, 1)

    if dr_score_values is not None:
        score_arr = np.asarray(dr_score_values, dtype=float)
        score_weight = (
            (score_arr - config["score_center"])
            / config["score_range"]
        )
        score_weight = np.clip(score_weight, 0, 1)
    else:
        score_weight = np.zeros_like(rank_weight)

    w = (
        config["score_weight"] * score_weight
        + config["rank_weight"] * rank_weight
    )

    return np.clip(w, 0, 1)


def add_optional_pssr_prediction(daily_df, pssr_artifact):
    if pssr_artifact is None:
        print("[INFO] pssr artifact가 없어 PSSR prediction을 건너뜁니다.")
        return daily_df, None

    daily_df = daily_df.copy()

    # 날짜 파생
    daily_df["month"] = pd.to_datetime(daily_df["date"]).dt.month
    daily_df["weekday"] = pd.to_datetime(daily_df["date"]).dt.weekday
    daily_df["is_weekend"] = (daily_df["weekday"] >= 5).astype(int)
    daily_df["month_sin"] = np.sin(2 * np.pi * daily_df["month"] / 12)
    daily_df["month_cos"] = np.cos(2 * np.pi * daily_df["month"] / 12)
    daily_df["weekday_sin"] = np.sin(2 * np.pi * daily_df["weekday"] / 7)
    daily_df["weekday_cos"] = np.cos(2 * np.pi * daily_df["weekday"] / 7)

    feature_cols = pssr_artifact.get("feature_cols", pssr_features)
    medians = pssr_artifact.get("feature_medians", {})

    X = pd.DataFrame(index=daily_df.index)

    for col in feature_cols:
        if col in daily_df.columns:
            X[col] = pd.to_numeric(daily_df[col], errors="coerce")
        else:
            X[col] = np.nan

        fill_value = medians.get(col, 0.0)
        try:
            fill_value = float(fill_value)
        except Exception:
            fill_value = 0.0

        X[col] = X[col].replace([np.inf, -np.inf], np.nan).fillna(fill_value)

    if "q75_model" in pssr_artifact and "q90_model" in pssr_artifact:
        q75_model = pssr_artifact["q75_model"]
        q90_model = pssr_artifact["q90_model"]

        q75_pred = np.clip(q75_model.predict(X[feature_cols]), 0, None)
        q90_pred = np.clip(q90_model.predict(X[feature_cols]), 0, None)

        dr_score_values = None
        if "dr_economic_score" in daily_df.columns:
            dr_score_values = daily_df["dr_economic_score"].values

        hybrid_w = compute_pssr_hybrid_weight_for_prediction(
            q75_pred=q75_pred,
            train_q75_pred=pssr_artifact.get("train_q75_pred", q75_pred),
            dr_score_values=dr_score_values,
            config=pssr_artifact.get("blend_config", None),
        )

        pred = (1 - hybrid_w) * q75_pred + hybrid_w * q90_pred

        daily_df["pssr_q75_pred"] = q75_pred
        daily_df["pssr_q90_pred"] = q90_pred
        daily_df["pssr_hybrid_weight"] = hybrid_w
        daily_df["pssr_reference_pred"] = np.clip(pred, 0, None)

    elif "model" in pssr_artifact:
        model = pssr_artifact["model"]
        pred = model.predict(X[feature_cols])
        daily_df["pssr_reference_pred"] = np.clip(pred, 0, None)

    else:
        raise ValueError("지원하지 않는 pssr_artifact 구조입니다.")

    return daily_df, X[feature_cols]


def read_json_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def parse_note_json(x):
    if x is None or pd.isna(x):
        return {}
    if isinstance(x, dict):
        return x
    try:
        return json.loads(x)
    except Exception:
        return {}


def get_table_columns(engine, table_name):
    q = text("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_name = :table_name
        ORDER BY ordinal_position
    """)
    return pd.read_sql(q, engine, params={"table_name": table_name})["column_name"].tolist()

In [ ]:
# =========================
# Load model_registry + Artifacts
# =========================

engine = get_engine()
test_db_connection(engine)

TARGET_MODEL_IDS = ["model1", "model2", "smp_combined", "dr_score", "pssr_p75"]

if MODEL_VERSION is None:
    registry_sql = text(f"""
        SELECT *
        FROM {MODEL_REGISTRY_TABLE}
        WHERE model_id = ANY(:model_ids)
          AND is_active = TRUE
        ORDER BY model_id, trained_at DESC
    """)
    registry_raw = pd.read_sql(
        registry_sql,
        engine,
        params={"model_ids": TARGET_MODEL_IDS}
    )
else:
    registry_sql = text(f"""
        SELECT *
        FROM {MODEL_REGISTRY_TABLE}
        WHERE model_id = ANY(:model_ids)
          AND version = :version
        ORDER BY model_id, trained_at DESC
    """)
    registry_raw = pd.read_sql(
        registry_sql,
        engine,
        params={"model_ids": TARGET_MODEL_IDS, "version": MODEL_VERSION}
    )

if registry_raw.empty:
    raise ValueError("model_registry에서 활성 모델을 찾지 못했습니다.")

registry = (
    registry_raw
    .sort_values(["model_id", "trained_at"], ascending=[True, False])
    .drop_duplicates("model_id", keep="first")
    .set_index("model_id")
)

display(registry.reset_index()[["model_id", "version", "model_path", "feature_count", "is_active"]])

required = ["model1", "model2", "smp_combined", "dr_score"]
missing = [m for m in required if m not in registry.index]
if missing:
    raise ValueError(f"필수 모델이 model_registry에 없습니다: {missing}")


def load_artifact_bytes_from_db(engine, model_id, version=None, artifact_type="model"):
    if version is None:
        sql = text("""
            SELECT artifact_bytes, artifact_format, version
            FROM model_artifacts
            WHERE model_id = :model_id
              AND artifact_type = :artifact_type
              AND is_active = TRUE
            ORDER BY created_at DESC
            LIMIT 1
        """)
        params = {
            "model_id": model_id,
            "artifact_type": artifact_type,
        }
    else:
        sql = text("""
            SELECT artifact_bytes, artifact_format, version
            FROM model_artifacts
            WHERE model_id = :model_id
              AND version = :version
              AND artifact_type = :artifact_type
              AND is_active = TRUE
            LIMIT 1
        """)
        params = {
            "model_id": model_id,
            "version": version,
            "artifact_type": artifact_type,
        }

    with engine.connect() as conn:
        row = conn.execute(sql, params).fetchone()

    if row is None:
        raise FileNotFoundError(
            f"No artifact found: model_id={model_id}, version={version}, artifact_type={artifact_type}"
        )

    return bytes(row[0]), row[1], row[2]


def load_joblib_artifact_from_db(engine, model_id, version=None, artifact_type="model"):
    artifact_bytes, artifact_format, resolved_version = load_artifact_bytes_from_db(
        engine=engine,
        model_id=model_id,
        version=version,
        artifact_type=artifact_type,
    )

    if artifact_format != "joblib":
        raise ValueError(f"{model_id}/{artifact_type} expected joblib, got {artifact_format}")

    return joblib.load(io.BytesIO(artifact_bytes)), resolved_version


def load_json_artifact_from_db(engine, model_id, version=None, artifact_type="features"):
    artifact_bytes, artifact_format, resolved_version = load_artifact_bytes_from_db(
        engine=engine,
        model_id=model_id,
        version=version,
        artifact_type=artifact_type,
    )

    if artifact_format != "json":
        raise ValueError(f"{model_id}/{artifact_type} expected json, got {artifact_format}")

    return json.loads(artifact_bytes.decode("utf-8")), resolved_version


model1, model1_version = load_joblib_artifact_from_db(
    engine, "model1", version=MODEL_VERSION, artifact_type="model"
)

model2, model2_version = load_joblib_artifact_from_db(
    engine, "model2", version=MODEL_VERSION, artifact_type="model"
)

combined_artifact, combined_version = load_joblib_artifact_from_db(
    engine, "smp_combined", version=MODEL_VERSION, artifact_type="pipeline"
)

dr_score_artifact, dr_version = load_joblib_artifact_from_db(
    engine, "dr_score", version=MODEL_VERSION, artifact_type="artifact"
)

pssr_artifact, pssr_version = load_joblib_artifact_from_db(
    engine, "pssr_p75", version=MODEL_VERSION, artifact_type="artifact"
)

reserve_artifact, reserve_version = load_joblib_artifact_from_db(
    engine, "reserve_power", version=MODEL_VERSION, artifact_type="artifact"
)

model1_features, _ = load_json_artifact_from_db(
    engine, "model1", version=model1_version, artifact_type="features"
)

model2_features, _ = load_json_artifact_from_db(
    engine, "model2", version=model2_version, artifact_type="features"
)

pssr_features, _ = load_json_artifact_from_db(
    engine, "pssr_p75", version=pssr_version, artifact_type="features"
)

reserve_features, _ = load_json_artifact_from_db(
    engine, "reserve_power", version=reserve_version, artifact_type="features"
)

model_version = combined_version

print("Loaded model version:", model_version)
print("model1 features:", len(model1_features))
print("model2 features:", len(model2_features))
print("DR artifact keys:", list(dr_score_artifact.keys()))
if pssr_artifact is not None:
    print("PSSR artifact keys:", list(pssr_artifact.keys()))

In [ ]:
# =========================
# Load model_features from DB
# =========================

feature_sql = text(f"""
    SELECT *
    FROM {MODEL_FEATURES_TABLE}
    WHERE model_id = :model_id
    ORDER BY datetime
""")

df_base = pd.read_sql(
    feature_sql,
    engine,
    params={"model_id": FEATURE_MODEL_ID}
)

if df_base.empty:
    raise ValueError("model_features에서 데이터를 불러오지 못했습니다.")

df_base[DATETIME_COL] = to_naive_datetime_series(df_base[DATETIME_COL])
df_base = df_base.dropna(subset=[DATETIME_COL]).copy()
df_base[DATETIME_COL] = df_base[DATETIME_COL].dt.floor("h")
df_base = (
    df_base
    .sort_values(DATETIME_COL)
    .drop_duplicates(DATETIME_COL, keep="last")
    .reset_index(drop=True)
)

print("model_features shape:", df_base.shape)
print("period:", df_base[DATETIME_COL].min(), "~", df_base[DATETIME_COL].max())
print("actual SMP period:", df_base.loc[df_base[TARGET_COL].notna(), DATETIME_COL].min(), "~", df_base.loc[df_base[TARGET_COL].notna(), DATETIME_COL].max())
display(df_base.head())
display(df_base.tail())

In [ ]:
# =========================
# Prediction Window
# =========================

actual_smp_df = df_base[df_base[TARGET_COL].notna()].copy()
if actual_smp_df.empty:
    raise ValueError("model_features에 실제 smp 값이 없습니다. recursive lag 계산을 위해 과거 smp가 필요합니다.")

latest_actual_dt = actual_smp_df[DATETIME_COL].max()


def resolve_run_now_kst(run_date_kst=None):
    """예측 실행 기준 시각을 KST 기준 Timestamp로 반환."""
    if run_date_kst is None or str(run_date_kst).strip() == "":
        return pd.Timestamp.now(tz="Asia/Seoul")

    ts = pd.Timestamp(run_date_kst)
    if ts.tzinfo is None:
        ts = ts.tz_localize("Asia/Seoul")
    else:
        ts = ts.tz_convert("Asia/Seoul")
    return ts


if PRED_START is not None:
    pred_start = to_naive_timestamp(PRED_START)
else:
    mode = str(PRED_START_MODE).strip().lower()

    if mode == "tomorrow_kst":
        # 서비스 운영 기준: 수집/실행일 다음날 00시부터 예측
        run_now_kst = resolve_run_now_kst(PRED_RUN_DATE_KST)
        pred_start = (run_now_kst.normalize() + pd.Timedelta(days=1)).tz_localize(None).floor("h")

    elif mode == "today_kst":
        # 실행일 당일 00시부터 예측
        run_now_kst = resolve_run_now_kst(PRED_RUN_DATE_KST)
        pred_start = run_now_kst.normalize().tz_localize(None).floor("h")

    elif mode == "after_latest_actual_hour":
        # 마지막 실제 SMP 다음 1시간부터 예측
        pred_start = (latest_actual_dt + pd.Timedelta(hours=1)).floor("h")

    elif mode == "after_latest_actual_day":
        # 기존 방식: 마지막 실제 SMP 날짜의 다음날 00시부터 예측
        pred_start = (latest_actual_dt.normalize() + pd.Timedelta(days=1)).floor("h")

    else:
        raise ValueError(
            f"지원하지 않는 PRED_START_MODE입니다: {PRED_START_MODE}. "
            "tomorrow_kst, today_kst, after_latest_actual_hour, after_latest_actual_day 중 하나를 사용하세요."
        )

pred_datetimes = pd.date_range(
    start=pred_start,
    periods=HORIZON_HOURS,
    freq="h"
)

pred_end = pred_datetimes[-1]

print("latest actual SMP datetime:", latest_actual_dt)
print("PRED_START_MODE:", PRED_START_MODE)
print("PRED_RUN_DATE_KST:", PRED_RUN_DATE_KST)
print("prediction start:", pred_start)
print("prediction end  :", pred_end)
print("horizon hours   :", len(pred_datetimes))


In [ ]:
# =========================
# Feature Construction Helpers
# =========================

# 시간 index 기반 조회용 테이블
df_by_dt = df_base.set_index(DATETIME_COL).sort_index()

# 전체 feature 후보
all_model_features = sorted(set(model1_features + model2_features))

# historical medians
historical_medians = {}
for col in all_model_features:
    if col in df_base.columns and pd.api.types.is_numeric_dtype(df_base[col]):
        historical_medians[col] = to_db_value(pd.to_numeric(df_base[col], errors="coerce").median())
    else:
        historical_medians[col] = 0.0

# 실제 SMP map
# 예측 시작 시점 이후의 실제 SMP는 recursive lag 계산에 사용하지 않는다.
# 예) 2026-07-06 실행 → 2026-07-07부터 예측할 때, 7/7 이후 SMP가 DB에 있어도 정답 누수를 막는다.
actual_smp_map = (
    df_base
    .loc[df_base[DATETIME_COL] < pred_start]
    .dropna(subset=[TARGET_COL])
    .set_index(DATETIME_COL)[TARGET_COL]
    .astype(float)
    .to_dict()
)

if len(actual_smp_map) == 0:
    raise ValueError("pred_start 이전의 실제 SMP가 없어 recursive lag 계산을 수행할 수 없습니다.")

predicted_smp_map = {}


def get_existing_feature_value(ts, col):
    """model_features에 해당 시각 row가 있으면 그 값을 우선 사용."""
    if col not in df_base.columns:
        return np.nan

    ts = pd.Timestamp(ts).floor("h")

    if ts in df_by_dt.index:
        value = df_by_dt.loc[ts, col]
        if isinstance(value, pd.Series):
            value = value.iloc[-1]
        if pd.notna(value):
            return value

    return np.nan


def fallback_feature_value(ts, col):
    """
    미래 row에 해당 feature가 없을 때 보수적 fallback.
    1) 같은 시각 1주 전
    2) 같은 시각 1일 전
    3) 예측시점 이전 마지막 관측값
    4) historical median
    """
    if col not in df_base.columns:
        return historical_medians.get(col, 0.0)

    ts = pd.Timestamp(ts).floor("h")

    for delta in [pd.Timedelta(days=7), pd.Timedelta(days=1)]:
        prev_ts = ts - delta
        if prev_ts in df_by_dt.index:
            value = df_by_dt.loc[prev_ts, col]
            if isinstance(value, pd.Series):
                value = value.iloc[-1]
            if pd.notna(value):
                return value

    past = df_base.loc[df_base[DATETIME_COL] < ts, [DATETIME_COL, col]].dropna()
    if len(past) > 0:
        return past.iloc[-1][col]

    return historical_medians.get(col, 0.0)


def get_feature_value(ts, col):
    value = get_existing_feature_value(ts, col)
    if pd.notna(value):
        return value
    return fallback_feature_value(ts, col)


def get_smp_value(ts):
    """
    Recursive SMP 조회.
    - 과거: 실제 SMP 사용
    - 미래 예측 완료 시각: 예측 SMP 사용
    - 그 외 결측: 같은 시각 1주 전 / 1일 전 / 마지막 실제값
    """
    ts = pd.Timestamp(ts).floor("h")

    if ts in predicted_smp_map:
        return float(predicted_smp_map[ts])

    if ts in actual_smp_map and pd.notna(actual_smp_map[ts]):
        return float(actual_smp_map[ts])

    for delta in [pd.Timedelta(days=7), pd.Timedelta(days=1)]:
        prev_ts = ts - delta
        if prev_ts in predicted_smp_map:
            return float(predicted_smp_map[prev_ts])
        if prev_ts in actual_smp_map and pd.notna(actual_smp_map[prev_ts]):
            return float(actual_smp_map[prev_ts])

    past_actual = df_base.loc[df_base[DATETIME_COL] < ts, [DATETIME_COL, TARGET_COL]].dropna()
    if len(past_actual) > 0:
        return float(past_actual.iloc[-1][TARGET_COL])

    return float(pd.Series(actual_smp_map).median())


def is_smp_lag_col(col):
    return re.fullmatch(r"smp_lag(\d+)", str(col)) is not None


def is_smp_roll_col(col):
    return re.fullmatch(r"smp_roll_(mean|std|max|min)_(\d+)", str(col)) is not None


def compute_smp_lag(col, ts):
    lag = int(re.fullmatch(r"smp_lag(\d+)", str(col)).group(1))

    if lag == 1 and not USE_RECURSIVE_SMP_LAG1:
        return np.nan

    return get_smp_value(pd.Timestamp(ts) - pd.Timedelta(hours=lag))


def compute_smp_roll(col, ts):
    m = re.fullmatch(r"smp_roll_(mean|std|max|min)_(\d+)", str(col))
    stat = m.group(1)
    window = int(m.group(2))

    vals = [
        get_smp_value(pd.Timestamp(ts) - pd.Timedelta(hours=i))
        for i in range(1, window + 1)
    ]

    vals = pd.Series(vals).dropna().astype(float)

    if len(vals) == 0:
        return historical_medians.get(col, 0.0)

    if stat == "mean":
        return float(vals.mean())
    if stat == "std":
        return float(vals.std(ddof=0))
    if stat == "max":
        return float(vals.max())
    if stat == "min":
        return float(vals.min())

    return historical_medians.get(col, 0.0)


def build_model_feature_row(ts, feature_cols):
    row = {}

    for col in feature_cols:
        if is_smp_lag_col(col):
            value = compute_smp_lag(col, ts)
        elif is_smp_roll_col(col):
            value = compute_smp_roll(col, ts)
        else:
            value = get_feature_value(ts, col)

        value = pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]
        if pd.isna(value) or np.isinf(value):
            value = historical_medians.get(col, 0.0)

        row[col] = float(value)

    return row

actual_reserve_map = {}

if "operating_reserve_power" in df_base.columns:
    actual_reserve_map = (
        df_base
        .dropna(subset=["operating_reserve_power"])
        .set_index(DATETIME_COL)["operating_reserve_power"]
        .astype(float)
        .to_dict()
    )

predicted_reserve_map = {}


def get_reserve_power_value(ts):
    """
    Recursive reserve 조회.
    - 과거: 실제 operating_reserve_power 사용
    - 미래 예측 완료 시각: predicted_reserve_map 사용
    - 그 외 결측: 1주 전 / 1일 전 / 마지막 실제값 / median fallback
    """
    ts = pd.Timestamp(ts).floor("h")

    if ts in predicted_reserve_map:
        return float(predicted_reserve_map[ts])

    if ts in actual_reserve_map and pd.notna(actual_reserve_map[ts]):
        return float(actual_reserve_map[ts])

    for delta in [pd.Timedelta(days=7), pd.Timedelta(days=1)]:
        prev_ts = ts - delta

        if prev_ts in predicted_reserve_map:
            return float(predicted_reserve_map[prev_ts])

        if prev_ts in actual_reserve_map and pd.notna(actual_reserve_map[prev_ts]):
            return float(actual_reserve_map[prev_ts])

    if "operating_reserve_power" in df_base.columns:
        past = (
            df_base
            .loc[df_base[DATETIME_COL] < ts, [DATETIME_COL, "operating_reserve_power"]]
            .dropna()
        )

        if len(past) > 0:
            return float(past.iloc[-1]["operating_reserve_power"])

    return float(historical_medians.get("operating_reserve_power", 0.0))


def is_reserve_power_lag_col(col):
    return re.fullmatch(r"operating_reserve_power_lag(\d+)", str(col)) is not None


def compute_reserve_power_lag(col, ts):
    lag = int(re.fullmatch(r"operating_reserve_power_lag(\d+)", str(col)).group(1))
    lag_ts = pd.Timestamp(ts).floor("h") - pd.Timedelta(hours=lag)
    return get_reserve_power_value(lag_ts)


In [ ]:
# =========================
# Recursive SMP Prediction: 168 hours
# =========================

records = []
feature_rows_model1 = []
feature_rows_model2 = []

for ts in pred_datetimes:
    x1_dict = build_model_feature_row(ts, model1_features)
    x2_dict = build_model_feature_row(ts, model2_features)

    X1_one = pd.DataFrame([x1_dict], columns=model1_features)
    X2_one = pd.DataFrame([x2_dict], columns=model2_features)

    smp_pred_base = float(model1.predict(X1_one)[0])
    smp_pred_residual = float(model2.predict(X2_one)[0])
    smp_pred_final = smp_pred_base + smp_pred_residual

    # SMP는 음수가 될 가능성이 낮으므로 음수 방지
    smp_pred_final = float(max(0, smp_pred_final))

    predicted_smp_map[pd.Timestamp(ts).floor("h")] = smp_pred_final

    records.append({
        "datetime": pd.Timestamp(ts),
        "smp_pred_base": smp_pred_base,
        "smp_pred_residual": smp_pred_residual,
        "smp_pred_final": smp_pred_final,
    })

    feature_rows_model1.append(x1_dict)
    feature_rows_model2.append(x2_dict)

hourly_pred = pd.DataFrame(records)
X1_future = pd.DataFrame(feature_rows_model1, columns=model1_features)
X2_future = pd.DataFrame(feature_rows_model2, columns=model2_features)

#SMP SCORE 정규화
SMP_SCORE_WINDOW_DAYS = 183

smp_score_ref_start = pred_start - pd.Timedelta(days=SMP_SCORE_WINDOW_DAYS)

hist_smp_values = df_base.loc[
    (df_base[DATETIME_COL] >= smp_score_ref_start) &
    (df_base[DATETIME_COL] < pred_start) &
    (df_base[TARGET_COL].notna()),
    TARGET_COL
].astype(float)

if len(hist_smp_values) < 24 * 30:
    print("[WARNING] 최근 6개월 SMP reference가 부족하여 전체 과거 SMP로 fallback합니다.")
    hist_smp_values = df_base[TARGET_COL].dropna().astype(float)

hourly_pred["smp_score"] = percentile_score_from_train(
    hist_smp_values,
    hourly_pred["smp_pred_final"]
)


print("hourly prediction shape:", hourly_pred.shape)
display(hourly_pred.head(DISPLAY_N))
display(hourly_pred.tail(DISPLAY_N))

In [ ]:
def apply_reserve_conditional_correction(values, cond_table):
    values = np.asarray(values, dtype=float)
    corrections = np.zeros(len(values), dtype=float)

    table = pd.DataFrame(cond_table)
    if table.empty:
        return corrections

    for i, v in enumerate(values):
        matched = table[
            (v >= table["bin_left"]) &
            (v <= table["bin_right"])
        ]

        if len(matched) == 0:
            if "condition_mean" in table.columns:
                nearest_idx = (table["condition_mean"] - v).abs().idxmin()
            else:
                nearest_idx = (table["bin_left"] - v).abs().idxmin()
            corrections[i] = float(table.loc[nearest_idx, "correction"])
        else:
            corrections[i] = float(matched.iloc[0]["correction"])

    return corrections



def predict_reserve_power_weekly(
    reserve_artifact,
    pred_datetimes,
    build_feature_func,
):
    model = reserve_artifact["base_model"]
    feature_cols = reserve_artifact["feature_cols"]
    medians = reserve_artifact.get("feature_medians", {})

    blend_cfg = reserve_artifact.get("blend", {})
    recent_cfg = reserve_artifact.get("recent_residual", {})
    cond_cfg = reserve_artifact.get("conditional_residual", {})

    blend_feature = blend_cfg.get("blend_feature", "operating_reserve_power_lag24")
    blend_weight = float(blend_cfg.get("blend_weight", 0.90))

    window_hours = int(recent_cfg.get("window_hours", 28 * 24))
    residual_history = pd.DataFrame(recent_cfg.get("history", []))

    if not residual_history.empty:
        residual_history["datetime"] = pd.to_datetime(
            residual_history["datetime"],
            errors="coerce"
        )
        residual_history["residual"] = pd.to_numeric(
            residual_history["residual"],
            errors="coerce"
        )
        residual_history = residual_history.dropna(subset=["datetime", "residual"])

    # 새 예측 run마다 reserve recursive map 초기화
    global predicted_reserve_map
    predicted_reserve_map = {}

    records = []
    feature_rows = []

    for ts in pred_datetimes:
        ts = pd.Timestamp(ts).floor("h")

        # 기본 row는 기존 feature builder로 생성
        row = build_feature_func(ts, feature_cols)

        # 예비력 lag 컬럼은 반드시 recursive 방식으로 덮어쓰기
        for col in feature_cols:
            if is_reserve_power_lag_col(col):
                row[col] = compute_reserve_power_lag(col, ts)

        # 결측 보정
        for col in feature_cols:
            if col not in row or pd.isna(row[col]):
                row[col] = medians.get(col, 0.0)

        X_one = pd.DataFrame([row], columns=feature_cols)
        X_one = X_one.apply(pd.to_numeric, errors="coerce")

        for col in feature_cols:
            fill_value = medians.get(col, 0.0)
            try:
                fill_value = float(fill_value)
            except Exception:
                fill_value = 0.0

            X_one[col] = (
                X_one[col]
                .replace([np.inf, -np.inf], np.nan)
                .fillna(fill_value)
            )

        # 1. base LGBM prediction
        base_pred = float(model.predict(X_one)[0])

        # 2. lag24 blend
        if blend_feature in X_one.columns:
            lag24_value = float(X_one[blend_feature].iloc[0])
        else:
            lag24_value = base_pred

        blend_pred = (1 - blend_weight) * base_pred + blend_weight * lag24_value

        # 3. recent residual adjustment
        recent_adj = 0.0

        if not residual_history.empty:
            window_start = ts - pd.Timedelta(hours=window_hours)

            hist_window = residual_history[
                (residual_history["datetime"] < ts) &
                (residual_history["datetime"] >= window_start)
            ]

            if len(hist_window) > 0:
                recent_adj = float(hist_window["residual"].mean())

        recent_pred = blend_pred + recent_adj

        # 4. conditional residual adjustment
        cond_feature = cond_cfg.get(
            "condition_feature",
            "operating_reserve_power_lag168"
        )
        cond_table = cond_cfg.get("correction_table", [])

        if cond_feature in X_one.columns:
            cond_value = float(X_one[cond_feature].iloc[0])
        else:
            cond_value = np.nan

        cond_adj = apply_reserve_conditional_correction(
            [cond_value],
            cond_table
        )[0]

        final_pred = max(0.0, recent_pred + cond_adj)

        records.append({
            "datetime": ts,
            "reserve_base_pred": base_pred,
            "reserve_blend_pred": blend_pred,
            "reserve_recent_adj": recent_adj,
            "reserve_cond_adj": cond_adj,
            "reserve_power_pred": final_pred,
        })

        feature_rows.append(row)

        # 핵심: 이번 시점 예측값을 이후 lag 계산에 사용
        predicted_reserve_map[ts] = final_pred

    reserve_pred_df = pd.DataFrame(records)
    X_reserve_future = pd.DataFrame(feature_rows, columns=feature_cols)

    return reserve_pred_df, X_reserve_future

In [ ]:
# =========================
# Build Daily Features for DR Score / Optional PSSR
# =========================

def get_daily_source_rows(date_value):
    date_value = pd.to_datetime(date_value).date()
    tmp = df_base[df_base[DATETIME_COL].dt.date == date_value].copy()
    return tmp


def daily_nbtp_value(date_value):
    tmp = get_daily_source_rows(date_value)

    candidates = []
    for col in ["nbtp_accepted", "nbtp_bid", "nbtp_mainland"]:
        if col in tmp.columns:
            vals = pd.to_numeric(tmp[col], errors="coerce").dropna()
            if len(vals) > 0:
                candidates.append(float(vals.mean()))

    if len(candidates) > 0:
        return candidates[0]

    # fallback: 1주 전, 1일 전
    for days in [7, 1]:
        prev_date = pd.to_datetime(date_value) - pd.Timedelta(days=days)
        tmp_prev = get_daily_source_rows(prev_date)
        for col in ["nbtp_accepted", "nbtp_bid", "nbtp_mainland"]:
            if col in tmp_prev.columns:
                vals = pd.to_numeric(tmp_prev[col], errors="coerce").dropna()
                if len(vals) > 0:
                    return float(vals.mean())

    return np.nan


def demand_proxy_from_rows(rows):
    demand_cols = [c for c in ["jlfd", "slfd", "mlfd", "forecast_load"] if c in rows.columns]
    if len(demand_cols) == 0 or len(rows) == 0:
        return pd.Series(dtype=float)
    return rows[demand_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)


def temp_proxy_from_rows(rows):
    temp_cols = [c for c in ["avg_temp_c", "temp_c_avg", "temp_c", "temperature"] if c in rows.columns]
    if len(temp_cols) == 0 or len(rows) == 0:
        return pd.Series(dtype=float)
    return rows[temp_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)


def build_daily_features_from_hourly(hourly_df):
    daily_rows = []

    tmp_hourly = hourly_df.copy()
    tmp_hourly["date"] = pd.to_datetime(tmp_hourly["datetime"]).dt.date

    for date_value, g in tmp_hourly.groupby("date"):
        g = g.sort_values("datetime").copy()
        source_rows = get_daily_source_rows(date_value)

        smp = g["smp_pred_final"].astype(float)
        base = g["smp_pred_base"].astype(float)
        residual = g["smp_pred_residual"].astype(float)

        nbtp = daily_nbtp_value(date_value)
        if pd.isna(nbtp):
            # nbtp가 없으면 historical median으로 대체
            nbtp = float(pd.to_numeric(
                df_base[[c for c in ["nbtp_accepted", "nbtp_bid"] if c in df_base.columns]].stack(),
                errors="coerce"
            ).median()) if any(c in df_base.columns for c in ["nbtp_accepted", "nbtp_bid"]) else float(smp.median())

        margin = smp - nbtp
        pos_margin = margin.clip(lower=0)

        demand_proxy = demand_proxy_from_rows(source_rows)
        temp_proxy = temp_proxy_from_rows(source_rows)

        row = {
            "date": pd.to_datetime(date_value),
            "pred_smp_mean": float(smp.mean()),
            "pred_smp_max": float(smp.max()),
            "pred_smp_p90": float(smp.quantile(0.90)),
            "pred_smp_std": float(smp.std(ddof=0)),
            "pred_smp_range": float(smp.max() - smp.min()),
            "pred_smp_over_120_hours": int((smp >= 120).sum()),
            "pred_smp_over_140_hours": int((smp >= 140).sum()),
            "pred_smp_over_160_hours": int((smp >= 160).sum()),

            "model1_pred_max": float(base.max()),
            "model1_pred_p90": float(base.quantile(0.90)),
            "model2_residual_mean": float(residual.mean()),
            "model2_residual_max": float(residual.max()),
            "model2_abs_residual_mean": float(residual.abs().mean()),
            "model2_positive_residual_sum": float(residual.clip(lower=0).sum()),

            "nbtp_mainland": float(nbtp),
            "smp_over_nbtp_hours": int((margin > 0).sum()),
            "smp_nbtp_positive_margin_sum": float(pos_margin.sum()),
            "smp_nbtp_margin_max": float(margin.max()),
            "smp_nbtp_margin_max_pos": float(max(0, margin.max())),
        }

        if len(demand_proxy) > 0:
            row["demand_proxy_mean"] = float(demand_proxy.mean())
            row["demand_proxy_max"] = float(demand_proxy.max())

        if len(temp_proxy) > 0:
            row["temp_proxy_mean"] = float(temp_proxy.mean())
            row["temp_proxy_max"] = float(temp_proxy.max())

        daily_rows.append(row)

    return pd.DataFrame(daily_rows)


daily_pred = build_daily_features_from_hourly(hourly_pred)

print("daily feature prediction shape:", daily_pred.shape)
display(daily_pred)

In [ ]:
# =========================
# DR Economic Score: 0~100 Numeric Score
# =========================

def make_historical_hourly_predictions_for_calibration(max_rows=None):
    """
    dr_score artifact에 percentile train distribution이 없을 때 fallback용.
    historical model_features에 대해 model1/model2 예측을 만들고 일별 feature 분포를 재구성한다.
    """
    hist = df_base[df_base[DATETIME_COL] < pred_start].copy()
    hist = hist.dropna(subset=[TARGET_COL]).copy()

    if max_rows is not None and len(hist) > max_rows:
        hist = hist.tail(max_rows).copy()

    # feature fill
    X1_hist = hist.reindex(columns=model1_features).copy()
    X2_hist = hist.reindex(columns=model2_features).copy()

    for col in model1_features:
        if col not in X1_hist.columns:
            X1_hist[col] = historical_medians.get(col, 0.0)
        X1_hist[col] = pd.to_numeric(X1_hist[col], errors="coerce").fillna(historical_medians.get(col, 0.0))

    for col in model2_features:
        if col not in X2_hist.columns:
            X2_hist[col] = historical_medians.get(col, 0.0)
        X2_hist[col] = pd.to_numeric(X2_hist[col], errors="coerce").fillna(historical_medians.get(col, 0.0))

    base_pred = model1.predict(X1_hist[model1_features])
    residual_pred = model2.predict(X2_hist[model2_features])
    final_pred = np.clip(base_pred + residual_pred, 0, None)

    hist_hourly = pd.DataFrame({
        "datetime": hist[DATETIME_COL].values,
        "smp_pred_base": base_pred,
        "smp_pred_residual": residual_pred,
        "smp_pred_final": final_pred,
    })

    return hist_hourly


def get_weight_map(dr_artifact):
    weight_df = pd.DataFrame(dr_artifact.get("learned_weight_df", []))
    if weight_df.empty:
        raise ValueError("dr_score artifact에 learned_weight_df가 없습니다.")
    return dict(zip(weight_df["feature"], weight_df["weight"]))


def add_dr_score(daily_df, dr_artifact):
    score_feature_cols = dr_artifact.get("score_feature_cols", [])
    weight_map = get_weight_map(dr_artifact)

    if len(score_feature_cols) == 0:
        raise ValueError("dr_score artifact에 score_feature_cols가 없습니다.")

    daily_df = daily_df.copy()

    # Artifact에 train distribution이 있으면 사용
    feature_train_values = dr_artifact.get("feature_train_values", None)
    raw_score_train_values = dr_artifact.get("raw_score_train_values", None)

    # 없으면 historical model prediction으로 calibration 분포 재구성
    if feature_train_values is None or raw_score_train_values is None:
        print("[INFO] DR artifact에 train distribution이 없어 historical calibration을 재구성합니다.")
        hist_hourly = make_historical_hourly_predictions_for_calibration()
        hist_daily = build_daily_features_from_hourly(hist_hourly)

        feature_train_values = {}
        for col in score_feature_cols:
            if col in hist_daily.columns:
                feature_train_values[col] = hist_daily[col].dropna().astype(float).tolist()
            else:
                feature_train_values[col] = [0.0]

    # 필요한 feature가 없으면 calibration median으로 대체
    for col in score_feature_cols:
        if col not in daily_df.columns:
            train_vals = pd.Series(feature_train_values.get(col, [0.0])).dropna().astype(float)
            daily_df[col] = float(train_vals.median()) if len(train_vals) > 0 else 0.0

        daily_df[col] = pd.to_numeric(daily_df[col], errors="coerce")
        train_vals = pd.Series(feature_train_values.get(col, [0.0])).dropna().astype(float)
        fill_value = float(train_vals.median()) if len(train_vals) > 0 else 0.0
        daily_df[col] = daily_df[col].replace([np.inf, -np.inf], np.nan).fillna(fill_value)

    # component percentile score
    daily_df["dr_economic_score_raw"] = 0.0

    for col in score_feature_cols:
        comp_score = percentile_score_from_train(
            feature_train_values.get(col, [0.0]),
            daily_df[col]
        )
        daily_df[col + "_score"] = comp_score
        daily_df["dr_economic_score_raw"] += comp_score * float(weight_map.get(col, 0.0))

    daily_df["dr_economic_score_raw"] = daily_df["dr_economic_score_raw"].clip(0, 100)

    # raw score calibration
    if raw_score_train_values is None:
        # feature_train_values로 raw train 분포 생성
        # 길이가 feature마다 다를 수 있어 hist_daily가 있으면 더 정확하게 재계산
        try:
            hist_hourly = make_historical_hourly_predictions_for_calibration()
            hist_daily = build_daily_features_from_hourly(hist_hourly)
            hist_daily = add_dr_score_without_raw_normalization(
                hist_daily,
                score_feature_cols,
                feature_train_values,
                weight_map
            )
            raw_score_train_values = hist_daily["dr_economic_score_raw"].dropna().astype(float).tolist()
        except Exception:
            raw_score_train_values = daily_df["dr_economic_score_raw"].dropna().astype(float).tolist()

    daily_df["dr_economic_score"] = percentile_score_from_train(
        raw_score_train_values,
        daily_df["dr_economic_score_raw"]
    )

    daily_df["dr_economic_score"] = daily_df["dr_economic_score"].clip(0, 100)

    return daily_df


def add_dr_score_without_raw_normalization(daily_df, score_feature_cols, feature_train_values, weight_map):
    daily_df = daily_df.copy()

    for col in score_feature_cols:
        if col not in daily_df.columns:
            daily_df[col] = 0.0
        daily_df[col] = pd.to_numeric(daily_df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)

    daily_df["dr_economic_score_raw"] = 0.0
    for col in score_feature_cols:
        comp_score = percentile_score_from_train(feature_train_values.get(col, [0.0]), daily_df[col])
        daily_df[col + "_score"] = comp_score
        daily_df["dr_economic_score_raw"] += comp_score * float(weight_map.get(col, 0.0))

    daily_df["dr_economic_score_raw"] = daily_df["dr_economic_score_raw"].clip(0, 100)
    return daily_df


daily_pred = add_dr_score(daily_pred, dr_score_artifact)

print("[DR score output: numeric 0~100, not group classification]")
display(daily_pred[["date", "dr_economic_score_raw", "dr_economic_score", "pred_smp_mean", "pred_smp_max", "nbtp_mainland"]])

In [ ]:
def add_hourly_dr_score_from_daily_score(
    pred_df,
    daily_score_col="dr_daily_score",
    smp_score_col="smp_score",
    output_col="dr_score",
):
    df = pred_df.copy()
    df["date"] = pd.to_datetime(df["datetime"]).dt.date

    def minmax_by_day(s):
        s = s.astype(float)
        denom = s.max() - s.min()
        if denom == 0 or pd.isna(denom):
            return pd.Series(0.5, index=s.index)
        return (s - s.min()) / denom

    df["_absolute_smp_weight"] = (
        df[smp_score_col]
        .astype(float)
        .clip(0, 100)
        / 100.0
    )

    df["_within_day_smp_weight"] = (
        df.groupby("date")[smp_score_col]
        .transform(minmax_by_day)
        .astype(float)
        .clip(0, 1)
    )

    df[output_col] = (
        df[daily_score_col].astype(float)
        * (
            0.4
            + 0.4 * df["_absolute_smp_weight"]
            + 0.2 * df["_within_day_smp_weight"]
        )
    )

    df[output_col] = df[output_col].clip(0, 100)

    return df.drop(
        columns=[
            "date",
            "_absolute_smp_weight",
            "_within_day_smp_weight",
        ]
    )

In [ ]:
# =========================
# PSSR Reference Prediction
# =========================

daily_pred, X_pssr_future = add_optional_pssr_prediction(
    daily_pred,
    pssr_artifact
)

display_cols = ["date", "dr_economic_score", "pred_smp_mean", "pred_smp_max"]

if "pssr_reference_pred" in daily_pred.columns:
    display_cols.append("pssr_reference_pred")

display(daily_pred[display_cols])

In [ ]:
# =========================
# Merge Daily DR Score to Hourly Predictions
# =========================

hourly_output = hourly_pred.copy()
hourly_output["date"] = pd.to_datetime(hourly_output["datetime"]).dt.normalize()

reserve_pred_df, X_reserve_future = predict_reserve_power_weekly(
    reserve_artifact=reserve_artifact,
    pred_datetimes=pred_datetimes,
    build_feature_func=build_model_feature_row,
)

hourly_output = hourly_output.merge(
    reserve_pred_df[[
        "datetime",
        "reserve_base_pred",
        "reserve_blend_pred",
        "reserve_recent_adj",
        "reserve_cond_adj",
        "reserve_power_pred",
    ]],
    on="datetime",
    how="left",
)

daily_score_cols = ["date", "dr_economic_score"]
if "pssr_reference_pred" in daily_pred.columns:
    daily_score_cols.append("pssr_reference_pred")

hourly_output = hourly_output.merge(
    daily_pred[daily_score_cols],
    on="date",
    how="left"
)

# 일별 DR score 보관
hourly_output = hourly_output.rename(
    columns={"dr_economic_score": "dr_daily_score"}
)

# 시간별 DR score 생성
# 같은 날 안에서 smp_score가 높은 시간대에 더 높은 dr_score 부여
hourly_output = add_hourly_dr_score_from_daily_score(
    pred_df=hourly_output,
    daily_score_col="dr_daily_score",
    smp_score_col="smp_score",
    output_col="dr_score",
)

REDUCTION_KW_DEFAULT = 1000.0

hourly_output["reduction_kw"] = REDUCTION_KW_DEFAULT
hourly_output["reduction_kwh"] = hourly_output["reduction_kw"] * 1.0

if "pssr_reference_pred" in hourly_output.columns:
    hourly_output["potential_dr_revenue_per_1000kw"] = (
        hourly_output["reduction_kwh"]
        * hourly_output["pssr_reference_pred"].astype(float)
    )

    hourly_output["score_weighted_revenue_per_1000kw"] = (
        hourly_output["potential_dr_revenue_per_1000kw"]
        * hourly_output["dr_score"].astype(float)
        / 100.0
    )
else:
    hourly_output["potential_dr_revenue_per_1000kw"] = np.nan
    hourly_output["score_weighted_revenue_per_1000kw"] = np.nan

hourly_output["area_name"] = AREA_NAME
hourly_output["model_id"] = PRED_MODEL_ID
hourly_output["model_version"] = model_version
hourly_output["predicted_smp"] = hourly_output["smp_pred_final"]
hourly_output["created_at"] = pd.Timestamp.now(tz="Asia/Seoul")

# predictions 테이블 기준 컬럼
candidate_cols = [
    "datetime",
    "area_name",
    "model_id",
    "smp_pred_base",
    "smp_pred_residual",
    "smp_pred_final",
    "smp_score",
    "dr_score",
    "model_version",
    "created_at",
    # 테이블에 있으면 자동 저장
    "reserve_power_pred",
    "score_weighted_revenue_per_1000kw",
]

predictions_to_save = hourly_output[candidate_cols].copy()

print("predictions_to_save shape:", predictions_to_save.shape)
display(predictions_to_save.head(DISPLAY_N))
display(predictions_to_save.tail(DISPLAY_N))

# 날짜별로 dr_score가 실제로 달라지는지 확인
check_dr_variation = (
    hourly_output
    .assign(date=pd.to_datetime(hourly_output["datetime"]).dt.date)
    .groupby("date")
    .agg(
        dr_score_min=("dr_score", "min"),
        dr_score_max=("dr_score", "max"),
        dr_score_std=("dr_score", "std"),
        smp_score_min=("smp_score", "min"),
        smp_score_max=("smp_score", "max"),
    )
    .reset_index()
)

display(check_dr_variation)

In [ ]:
# =========================
# Save to predictions table
# =========================

# 테이블이 없으면 기본 스키마 생성
with engine.begin() as conn:
    conn.execute(text(f"""
        CREATE TABLE IF NOT EXISTS {PREDICTIONS_TABLE} (
            datetime TIMESTAMPTZ NOT NULL,
            area_name TEXT NOT NULL,
            model_id TEXT NOT NULL,
            smp_pred_base FLOAT,
            smp_pred_residual FLOAT,
            smp_pred_final FLOAT,
            smp_score FLOAT,
            dr_score FLOAT,
            model_version TEXT,
            created_at TIMESTAMPTZ DEFAULT NOW(),
            reserve_power_pred FLOAT,
            score_weighted_revenue_per_1000kw FLOAT,
            PRIMARY KEY (datetime, area_name, model_id)
        )
    """))

    # 기존 테이블이 있을 경우 필요한 컬럼 보강
    conn.execute(text(f"""
        ALTER TABLE {PREDICTIONS_TABLE}
        ADD COLUMN IF NOT EXISTS reserve_power_pred FLOAT,
        ADD COLUMN IF NOT EXISTS score_weighted_revenue_per_1000kw FLOAT
    """))

    # 기존 model_id 길이 문제가 있었던 경우 방어
    conn.execute(text(f"""
        ALTER TABLE {PREDICTIONS_TABLE}
        ALTER COLUMN model_id TYPE TEXT
    """))

table_cols = get_table_columns(engine, PREDICTIONS_TABLE)

required_prediction_cols = [
    "datetime",
    "area_name",
    "model_id",
    "smp_pred_base",
    "smp_pred_residual",
    "smp_pred_final",
    "smp_score",
    "dr_score",
    "model_version",
    "created_at",
    "reserve_power_pred",
    "score_weighted_revenue_per_1000kw",
]

missing_cols = [c for c in required_prediction_cols if c not in table_cols]

if missing_cols:
    raise ValueError(f"predictions 테이블에 필요한 컬럼이 없습니다: {missing_cols}")

insert_cols = [c for c in required_prediction_cols if c in predictions_to_save.columns]
save_df = predictions_to_save[insert_cols].copy()

# DB 호환 타입 정리
for col in save_df.columns:
    if col in ["datetime", "created_at"]:
        save_df[col] = pd.to_datetime(save_df[col], errors="coerce")
    elif col in ["area_name", "model_id", "model_version"]:
        save_df[col] = save_df[col].astype(str)
    else:
        save_df[col] = pd.to_numeric(save_df[col], errors="coerce")

print("Insert columns:", insert_cols)
print("Save rows:", len(save_df))

if SAVE_MODE == "delete_insert":
    start_dt = save_df["datetime"].min()
    end_dt = save_df["datetime"].max()

    with engine.begin() as conn:
        conn.execute(
            text(f"""
                DELETE FROM {PREDICTIONS_TABLE}
                WHERE area_name = :area_name
                  AND model_id = :model_id
                  AND datetime BETWEEN :start_dt AND :end_dt
            """),
            {
                "area_name": AREA_NAME,
                "model_id": PRED_MODEL_ID,
                "start_dt": start_dt,
                "end_dt": end_dt,
            }
        )

    save_df.to_sql(
        PREDICTIONS_TABLE,
        engine,
        if_exists="append",
        index=False,
        method="multi",
        chunksize=1000,
    )

else:
    raise ValueError(f"Unknown SAVE_MODE: {SAVE_MODE}")

print(f"Saved {len(save_df)} rows to {PREDICTIONS_TABLE}.")
print("period:", save_df["datetime"].min(), "~", save_df["datetime"].max())

In [ ]:
# =========================
# Verify Saved Rows
# =========================

verify_sql = text(f"""
    SELECT *
    FROM {PREDICTIONS_TABLE}
    WHERE area_name = :area_name
      AND model_id = :model_id
      AND datetime BETWEEN :start_dt AND :end_dt
    ORDER BY datetime
""")

saved_check = pd.read_sql(
    verify_sql,
    engine,
    params={
        "area_name": AREA_NAME,
        "model_id": PRED_MODEL_ID,
        "start_dt": predictions_to_save["datetime"].min(),
        "end_dt": predictions_to_save["datetime"].max(),
    }
)

print("saved_check shape:", saved_check.shape)
display(saved_check.head(DISPLAY_N))
display(saved_check.tail(DISPLAY_N))

In [ ]:
PREDICTION_EXPLAIN_TABLE = "prediction_explain_values"

with engine.begin() as conn:
    conn.execute(text(f"""
        CREATE TABLE IF NOT EXISTS {PREDICTION_EXPLAIN_TABLE} (
            id BIGSERIAL PRIMARY KEY,
            prediction_datetime TIMESTAMPTZ NOT NULL,
            area_name TEXT NOT NULL,
            prediction_model_id TEXT NOT NULL,
            model_id TEXT NOT NULL,
            model_version TEXT,
            component TEXT NOT NULL,
            explain_type TEXT NOT NULL,
            feature_name TEXT NOT NULL,
            feature_value DOUBLE PRECISION,
            effect_value DOUBLE PRECISION,
            abs_effect_value DOUBLE PRECISION,
            effect_rank INT,
            base_value DOUBLE PRECISION,
            created_at TIMESTAMPTZ DEFAULT NOW()
        )
    """))

    conn.execute(text(f"""
        CREATE INDEX IF NOT EXISTS idx_prediction_explain_lookup
        ON {PREDICTION_EXPLAIN_TABLE}
        (prediction_datetime, area_name, prediction_model_id, model_id, component)
    """))

In [ ]:
model_explain_summary = pd.read_sql("""
    SELECT
        model_id,
        version,
        component,
        explain_type,
        COUNT(*) AS row_count,
        MIN(target_datetime) AS min_target_datetime,
        MAX(target_datetime) AS max_target_datetime,
        MIN(target_date) AS min_target_date,
        MAX(target_date) AS max_target_date
    FROM model_explain_values
    GROUP BY model_id, version, component, explain_type
    ORDER BY model_id, component, explain_type
""", engine)

display(model_explain_summary)

In [ ]:
# =========================
# Save Prediction-level SHAP / Contribution Values
# =========================

import shap

PREDICTION_EXPLAIN_TABLE = "prediction_explain_values"

PRED_SHAP_TOP_N = 10
PRED_POSTHOC_TOP_N = 3

# -------------------------------------------------
# 1. Ensure table
# -------------------------------------------------

with engine.begin() as conn:
    conn.execute(text(f"""
        CREATE TABLE IF NOT EXISTS {PREDICTION_EXPLAIN_TABLE} (
            id BIGSERIAL PRIMARY KEY,
            prediction_datetime TIMESTAMPTZ NOT NULL,
            area_name TEXT NOT NULL,
            prediction_model_id TEXT NOT NULL,
            model_id TEXT NOT NULL,
            model_version TEXT,
            component TEXT NOT NULL,
            explain_type TEXT NOT NULL,
            feature_name TEXT NOT NULL,
            feature_value DOUBLE PRECISION,
            effect_value DOUBLE PRECISION,
            abs_effect_value DOUBLE PRECISION,
            effect_rank INT,
            base_value DOUBLE PRECISION,
            created_at TIMESTAMPTZ DEFAULT NOW()
        )
    """))

    conn.execute(text(f"""
        CREATE INDEX IF NOT EXISTS idx_prediction_explain_lookup
        ON {PREDICTION_EXPLAIN_TABLE}
        (
            prediction_datetime,
            area_name,
            prediction_model_id,
            model_id,
            component
        )
    """))


# -------------------------------------------------
# 2. Utility functions
# -------------------------------------------------

def scalar_expected_value(explainer):
    ev = explainer.expected_value
    if isinstance(ev, (list, tuple, np.ndarray)):
        return float(np.ravel(ev)[0])
    return float(ev)


def normalize_shap_values(shap_values):
    arr = np.asarray(shap_values)

    # 일부 LightGBM/SHAP 버전에서 (n, features, 1) 형태가 나올 수 있음
    if arr.ndim == 3:
        arr = arr[:, :, 0]

    return arr


def clean_float_value(x):
    try:
        if x is None or pd.isna(x):
            return None
    except Exception:
        pass

    try:
        x = float(x)
        if np.isinf(x) or np.isnan(x):
            return None
        return x
    except Exception:
        return None


def build_prediction_effect_rows(
    X_data,
    effect_values,
    prediction_datetimes,
    model_id,
    model_version,
    component,
    explain_type,
    base_value=0.0,
    top_n=10,
):
    """
    X_data: shape (n_rows, n_features)
    effect_values: shape (n_rows, n_features)
    prediction_datetimes: length n_rows
    """
    rows = []

    X_data = X_data.copy().reset_index(drop=True)
    arr = normalize_shap_values(effect_values)

    prediction_datetimes = pd.Series(prediction_datetimes).reset_index(drop=True)

    if len(X_data) != len(arr):
        raise ValueError(
            f"X_data length와 effect_values length가 다릅니다: {len(X_data)} vs {len(arr)}"
        )

    for i in range(len(X_data)):
        tmp = pd.DataFrame({
            "feature_name": X_data.columns,
            "feature_value": X_data.iloc[i].values,
            "effect_value": arr[i],
        })

        tmp["abs_effect_value"] = tmp["effect_value"].abs()
        tmp = tmp.sort_values("abs_effect_value", ascending=False).head(top_n)
        tmp["effect_rank"] = range(1, len(tmp) + 1)

        pred_dt = pd.to_datetime(prediction_datetimes.iloc[i])

        for _, r in tmp.iterrows():
            rows.append({
                "prediction_datetime": pred_dt,
                "area_name": AREA_NAME,
                "prediction_model_id": PRED_MODEL_ID,
                "model_id": model_id,
                "model_version": model_version,
                "component": component,
                "explain_type": explain_type,
                "feature_name": str(r["feature_name"]),
                "feature_value": clean_float_value(r["feature_value"]),
                "effect_value": clean_float_value(r["effect_value"]),
                "abs_effect_value": clean_float_value(r["abs_effect_value"]),
                "effect_rank": int(r["effect_rank"]),
                "base_value": clean_float_value(base_value),
            })

    return rows


def build_combined_smp_prediction_rows(
    X1_data,
    X2_data,
    shap1,
    shap2,
    prediction_datetimes,
    base1,
    base2,
    top_n=10,
):
    rows = []

    X1_data = X1_data.copy().reset_index(drop=True)
    X2_data = X2_data.copy().reset_index(drop=True)

    s1 = normalize_shap_values(shap1)
    s2 = normalize_shap_values(shap2)

    prediction_datetimes = pd.Series(prediction_datetimes).reset_index(drop=True)

    if len(X1_data) != len(X2_data):
        raise ValueError("X1_data와 X2_data 길이가 다릅니다.")

    for i in range(len(X1_data)):
        shap_map = {}
        value_map = {}

        for feature, value, sv in zip(X1_data.columns, X1_data.iloc[i].values, s1[i]):
            shap_map[feature] = shap_map.get(feature, 0.0) + float(sv)
            value_map[feature] = value

        for feature, value, sv in zip(X2_data.columns, X2_data.iloc[i].values, s2[i]):
            shap_map[feature] = shap_map.get(feature, 0.0) + float(sv)
            value_map[feature] = value

        tmp = pd.DataFrame({
            "feature_name": list(shap_map.keys()),
            "feature_value": [value_map.get(k, np.nan) for k in shap_map.keys()],
            "effect_value": list(shap_map.values()),
        })

        tmp["abs_effect_value"] = tmp["effect_value"].abs()
        tmp = tmp.sort_values("abs_effect_value", ascending=False).head(top_n)
        tmp["effect_rank"] = range(1, len(tmp) + 1)

        pred_dt = pd.to_datetime(prediction_datetimes.iloc[i])

        for _, r in tmp.iterrows():
            rows.append({
                "prediction_datetime": pred_dt,
                "area_name": AREA_NAME,
                "prediction_model_id": PRED_MODEL_ID,
                "model_id": "smp_combined",
                "model_version": model_version,
                "component": "smp_final",
                "explain_type": "shap",
                "feature_name": str(r["feature_name"]),
                "feature_value": clean_float_value(r["feature_value"]),
                "effect_value": clean_float_value(r["effect_value"]),
                "abs_effect_value": clean_float_value(r["abs_effect_value"]),
                "effect_rank": int(r["effect_rank"]),
                "base_value": clean_float_value(base1 + base2),
            })

    return rows

In [ ]:
# =========================
# Compute + Insert Prediction-level Explanations
# =========================

prediction_explain_rows = []

prediction_datetimes = pd.to_datetime(hourly_output["datetime"]).reset_index(drop=True)

# 입력 feature 정리
X1_future_explain = X1_future[model1_features].copy().reset_index(drop=True)
X2_future_explain = X2_future[model2_features].copy().reset_index(drop=True)

for c in X1_future_explain.columns:
    X1_future_explain[c] = pd.to_numeric(X1_future_explain[c], errors="coerce").fillna(0.0)

for c in X2_future_explain.columns:
    X2_future_explain[c] = pd.to_numeric(X2_future_explain[c], errors="coerce").fillna(0.0)


# -------------------------------------------------
# 1. SMP model1 / model2 / combined SHAP
# -------------------------------------------------

explainer_model1 = shap.TreeExplainer(model1)
explainer_model2 = shap.TreeExplainer(model2)

shap_model1 = explainer_model1.shap_values(X1_future_explain)
shap_model2 = explainer_model2.shap_values(X2_future_explain)

base_model1 = scalar_expected_value(explainer_model1)
base_model2 = scalar_expected_value(explainer_model2)

prediction_explain_rows += build_prediction_effect_rows(
    X_data=X1_future_explain,
    effect_values=shap_model1,
    prediction_datetimes=prediction_datetimes,
    model_id="model1",
    model_version=model1_version,
    component="smp_base",
    explain_type="shap",
    base_value=base_model1,
    top_n=PRED_SHAP_TOP_N,
)

prediction_explain_rows += build_prediction_effect_rows(
    X_data=X2_future_explain,
    effect_values=shap_model2,
    prediction_datetimes=prediction_datetimes,
    model_id="model2",
    model_version=model2_version,
    component="smp_residual",
    explain_type="shap",
    base_value=base_model2,
    top_n=PRED_SHAP_TOP_N,
)

prediction_explain_rows += build_combined_smp_prediction_rows(
    X1_data=X1_future_explain,
    X2_data=X2_future_explain,
    shap1=shap_model1,
    shap2=shap_model2,
    prediction_datetimes=prediction_datetimes,
    base1=base_model1,
    base2=base_model2,
    top_n=PRED_SHAP_TOP_N,
)

print("SMP prediction explanations:", len(prediction_explain_rows))


# -------------------------------------------------
# 2. PSSR SHAP
#    PSSR는 일 단위 예측이므로 같은 날짜의 24시간에 복제 저장
# -------------------------------------------------

pssr_rows = []

if "X_pssr_future" in globals() and X_pssr_future is not None:
    X_pssr_daily = X_pssr_future.copy().reset_index(drop=True)
    daily_for_pssr = daily_pred.copy().reset_index(drop=True)
    daily_for_pssr["date_key"] = pd.to_datetime(daily_for_pssr["date"]).dt.date

    for c in X_pssr_daily.columns:
        X_pssr_daily[c] = pd.to_numeric(X_pssr_daily[c], errors="coerce").fillna(0.0)

    # hybrid artifact
    if "q75_model" in pssr_artifact and "q90_model" in pssr_artifact:
        q75_model = pssr_artifact["q75_model"]
        q90_model = pssr_artifact["q90_model"]

        pssr_explainer_q75 = shap.TreeExplainer(q75_model)
        pssr_explainer_q90 = shap.TreeExplainer(q90_model)

        shap_q75 = normalize_shap_values(pssr_explainer_q75.shap_values(X_pssr_daily))
        shap_q90 = normalize_shap_values(pssr_explainer_q90.shap_values(X_pssr_daily))

        base_q75 = scalar_expected_value(pssr_explainer_q75)
        base_q90 = scalar_expected_value(pssr_explainer_q90)

        q75_pred = np.clip(q75_model.predict(X_pssr_daily), 0, None)

        if "dr_economic_score" in daily_for_pssr.columns:
            dr_score_values = daily_for_pssr["dr_economic_score"].values
        else:
            dr_score_values = None

        hybrid_w = compute_pssr_hybrid_weight_for_prediction(
            q75_pred=q75_pred,
            train_q75_pred=pssr_artifact.get("train_q75_pred", q75_pred),
            dr_score_values=dr_score_values,
            config=pssr_artifact.get("blend_config", None),
        )

        pssr_shap_daily = (
            (1 - hybrid_w.reshape(-1, 1)) * shap_q75
            + hybrid_w.reshape(-1, 1) * shap_q90
        )

        pssr_base_daily = (
            (1 - hybrid_w) * base_q75
            + hybrid_w * base_q90
        )

    # old p75 artifact
    elif "model" in pssr_artifact:
        pssr_model = pssr_artifact["model"]
        pssr_explainer = shap.TreeExplainer(pssr_model)

        pssr_shap_daily = normalize_shap_values(
            pssr_explainer.shap_values(X_pssr_daily)
        )

        pssr_base_daily = np.repeat(
            scalar_expected_value(pssr_explainer),
            len(X_pssr_daily)
        )

    else:
        print("[WARNING] 지원하지 않는 PSSR artifact 구조라 PSSR SHAP 저장을 건너뜁니다.")
        pssr_shap_daily = None
        pssr_base_daily = None

    if pssr_shap_daily is not None:
        hourly_for_pssr = hourly_output[["datetime"]].copy()
        hourly_for_pssr["date_key"] = pd.to_datetime(hourly_for_pssr["datetime"]).dt.date

        for daily_i in range(len(X_pssr_daily)):
            date_key = daily_for_pssr.loc[daily_i, "date_key"]

            target_hours = hourly_for_pssr.loc[
                hourly_for_pssr["date_key"] == date_key,
                "datetime"
            ].tolist()

            if len(target_hours) == 0:
                continue

            # 일별 SHAP top-N 계산
            tmp = pd.DataFrame({
                "feature_name": X_pssr_daily.columns,
                "feature_value": X_pssr_daily.iloc[daily_i].values,
                "effect_value": pssr_shap_daily[daily_i],
            })

            tmp["abs_effect_value"] = tmp["effect_value"].abs()
            tmp = tmp.sort_values("abs_effect_value", ascending=False).head(PRED_SHAP_TOP_N)
            tmp["effect_rank"] = range(1, len(tmp) + 1)

            # 같은 날짜의 시간별 예측 row에 복제 저장
            for pred_dt in target_hours:
                for _, r in tmp.iterrows():
                    pssr_rows.append({
                        "prediction_datetime": pd.to_datetime(pred_dt),
                        "area_name": AREA_NAME,
                        "prediction_model_id": PRED_MODEL_ID,
                        "model_id": "pssr_p75",
                        "model_version": pssr_version,
                        "component": "pssr_reference",
                        "explain_type": "shap",
                        "feature_name": str(r["feature_name"]),
                        "feature_value": clean_float_value(r["feature_value"]),
                        "effect_value": clean_float_value(r["effect_value"]),
                        "abs_effect_value": clean_float_value(r["abs_effect_value"]),
                        "effect_rank": int(r["effect_rank"]),
                        "base_value": clean_float_value(pssr_base_daily[daily_i]),
                    })

prediction_explain_rows += pssr_rows

print("PSSR prediction explanations:", len(pssr_rows))


# -------------------------------------------------
# 3. Reserve SHAP + posthoc contribution
# -------------------------------------------------

reserve_rows = []

if "X_reserve_future" in globals() and X_reserve_future is not None:
    reserve_model = reserve_artifact["base_model"]
    reserve_feature_cols = reserve_artifact["feature_cols"]

    X_reserve_explain = X_reserve_future[reserve_feature_cols].copy().reset_index(drop=True)

    for c in X_reserve_explain.columns:
        X_reserve_explain[c] = pd.to_numeric(X_reserve_explain[c], errors="coerce").fillna(0.0)

    reserve_explainer = shap.TreeExplainer(reserve_model)
    reserve_shap = reserve_explainer.shap_values(X_reserve_explain)
    reserve_base = scalar_expected_value(reserve_explainer)

    reserve_rows += build_prediction_effect_rows(
        X_data=X_reserve_explain,
        effect_values=reserve_shap,
        prediction_datetimes=prediction_datetimes,
        model_id="reserve_power",
        model_version=reserve_version,
        component="reserve_base",
        explain_type="shap",
        base_value=reserve_base,
        top_n=PRED_SHAP_TOP_N,
    )

    # 후처리 contribution
    reserve_posthoc_df = pd.DataFrame({
        "lag24_blend_contribution": (
            reserve_pred_df["reserve_blend_pred"].values
            - reserve_pred_df["reserve_base_pred"].values
        ),
        "recent_residual_contribution": reserve_pred_df["reserve_recent_adj"].values,
        "conditional_residual_contribution": reserve_pred_df["reserve_cond_adj"].values,
    })

    reserve_posthoc_effect_values = reserve_posthoc_df.values

    reserve_rows += build_prediction_effect_rows(
        X_data=reserve_posthoc_df,
        effect_values=reserve_posthoc_effect_values,
        prediction_datetimes=prediction_datetimes,
        model_id="reserve_power",
        model_version=reserve_version,
        component="reserve_final",
        explain_type="posthoc_contribution",
        base_value=0.0,
        top_n=PRED_POSTHOC_TOP_N,
    )

prediction_explain_rows += reserve_rows

print("Reserve prediction explanations:", len(reserve_rows))


# -------------------------------------------------
# 4. DR score contribution
#    DR은 SHAP이 아니라 score contribution.
#    daily contribution을 hourly factor로 스케일링해서 시간별 row에 저장.
# -------------------------------------------------

dr_rows = []

try:
    weight_df = pd.DataFrame(dr_score_artifact.get("learned_weight_df", []))
    weight_map = dict(zip(weight_df["feature"], weight_df["weight"]))
    score_feature_cols = dr_score_artifact.get("score_feature_cols", [])

    dr_daily = daily_pred.copy().reset_index(drop=True)
    dr_daily["date_key"] = pd.to_datetime(dr_daily["date"]).dt.date

    hourly_dr = hourly_output[["datetime", "dr_score", "dr_daily_score", "smp_score"]].copy()
    hourly_dr["date_key"] = pd.to_datetime(hourly_dr["datetime"]).dt.date

    # hourly factor = hourly dr_score / daily dr score
    # daily score가 0이면 factor는 0 처리
    hourly_dr["hourly_dr_factor"] = np.where(
        hourly_dr["dr_daily_score"].astype(float).abs() > 1e-9,
        hourly_dr["dr_score"].astype(float) / hourly_dr["dr_daily_score"].astype(float),
        0.0
    )

    for daily_i in range(len(dr_daily)):
        date_key = dr_daily.loc[daily_i, "date_key"]

        # daily contribution 계산
        contrib_items = []

        for feature in score_feature_cols:
            score_col = feature + "_score"

            if score_col in dr_daily.columns:
                feature_score = float(dr_daily.loc[daily_i, score_col])
            else:
                feature_score = 0.0

            weight = float(weight_map.get(feature, 0.0))
            contribution = feature_score * weight

            feature_value = (
                dr_daily.loc[daily_i, feature]
                if feature in dr_daily.columns
                else np.nan
            )

            contrib_items.append({
                "feature_name": feature,
                "feature_value": feature_value,
                "daily_effect_value": contribution,
                "abs_daily_effect_value": abs(contribution),
            })

        contrib_tmp = pd.DataFrame(contrib_items)

        if contrib_tmp.empty:
            continue

        target_hours = hourly_dr[hourly_dr["date_key"] == date_key].copy()

        if len(target_hours) == 0:
            continue

        for _, h in target_hours.iterrows():
            factor = float(h["hourly_dr_factor"])

            tmp = contrib_tmp.copy()
            tmp["effect_value"] = tmp["daily_effect_value"] * factor
            tmp["abs_effect_value"] = tmp["effect_value"].abs()
            tmp = tmp.sort_values("abs_effect_value", ascending=False).head(PRED_SHAP_TOP_N)
            tmp["effect_rank"] = range(1, len(tmp) + 1)

            for _, r in tmp.iterrows():
                dr_rows.append({
                    "prediction_datetime": pd.to_datetime(h["datetime"]),
                    "area_name": AREA_NAME,
                    "prediction_model_id": PRED_MODEL_ID,
                    "model_id": "dr_score",
                    "model_version": dr_version,
                    "component": "economic_dr_score",
                    "explain_type": "score_contribution",
                    "feature_name": str(r["feature_name"]),
                    "feature_value": clean_float_value(r["feature_value"]),
                    "effect_value": clean_float_value(r["effect_value"]),
                    "abs_effect_value": clean_float_value(r["abs_effect_value"]),
                    "effect_rank": int(r["effect_rank"]),
                    "base_value": 0.0,
                })

except Exception as e:
    print("[WARNING] DR score contribution 저장 중 오류가 발생해 건너뜁니다:", e)

prediction_explain_rows += dr_rows

print("DR prediction explanations:", len(dr_rows))


# -------------------------------------------------
# 5. Delete existing rows for same prediction period, then insert
# -------------------------------------------------

prediction_explain_df = pd.DataFrame(prediction_explain_rows)

if prediction_explain_df.empty:
    print("[WARNING] 저장할 prediction explanation row가 없습니다.")
else:
    start_dt = pd.to_datetime(predictions_to_save["datetime"].min())
    end_dt = pd.to_datetime(predictions_to_save["datetime"].max())

    with engine.begin() as conn:
        conn.execute(
            text(f"""
                DELETE FROM {PREDICTION_EXPLAIN_TABLE}
                WHERE area_name = :area_name
                  AND prediction_model_id = :prediction_model_id
                  AND prediction_datetime BETWEEN :start_dt AND :end_dt
            """),
            {
                "area_name": AREA_NAME,
                "prediction_model_id": PRED_MODEL_ID,
                "start_dt": start_dt,
                "end_dt": end_dt,
            }
        )

    # 타입 정리
    prediction_explain_df["prediction_datetime"] = pd.to_datetime(
        prediction_explain_df["prediction_datetime"],
        errors="coerce"
    )

    str_cols = [
        "area_name",
        "prediction_model_id",
        "model_id",
        "model_version",
        "component",
        "explain_type",
        "feature_name",
    ]

    for c in str_cols:
        prediction_explain_df[c] = prediction_explain_df[c].astype(str)

    numeric_cols = [
        "feature_value",
        "effect_value",
        "abs_effect_value",
        "effect_rank",
        "base_value",
    ]

    for c in numeric_cols:
        prediction_explain_df[c] = pd.to_numeric(
            prediction_explain_df[c],
            errors="coerce"
        )

    prediction_explain_df.to_sql(
        PREDICTION_EXPLAIN_TABLE,
        engine,
        if_exists="append",
        index=False,
        method="multi",
        chunksize=1000,
    )

    print(f"Saved prediction explanations: {len(prediction_explain_df)} rows")
    display(
        prediction_explain_df
        .groupby(["model_id", "component", "explain_type"])
        .size()
        .reset_index(name="row_count")
    )

In [ ]:
# =========================
# Verify Prediction-level Explanation Rows
# =========================

pred_explain_summary = pd.read_sql(
    text(f"""
        SELECT
            prediction_model_id,
            model_id,
            model_version,
            component,
            explain_type,
            COUNT(*) AS row_count,
            COUNT(DISTINCT prediction_datetime) AS explained_hours,
            MIN(prediction_datetime) AS min_prediction_datetime,
            MAX(prediction_datetime) AS max_prediction_datetime
        FROM {PREDICTION_EXPLAIN_TABLE}
        WHERE area_name = :area_name
          AND prediction_model_id = :prediction_model_id
          AND prediction_datetime BETWEEN :start_dt AND :end_dt
        GROUP BY
            prediction_model_id,
            model_id,
            model_version,
            component,
            explain_type
        ORDER BY
            model_id,
            component,
            explain_type
    """),
    engine,
    params={
        "area_name": AREA_NAME,
        "prediction_model_id": PRED_MODEL_ID,
        "start_dt": predictions_to_save["datetime"].min(),
        "end_dt": predictions_to_save["datetime"].max(),
    }
)

display(pred_explain_summary)

expected_hours = len(predictions_to_save)
print("Expected prediction hours:", expected_hours)

display(
    pred_explain_summary.assign(
        hour_coverage_ratio=pred_explain_summary["explained_hours"] / expected_hours
    )
)

In [ ]:
# =========================
# Inspect One Prediction Datetime
# =========================

one_dt = pd.to_datetime(predictions_to_save["datetime"].min())

one_prediction_explain = pd.read_sql(
    text(f"""
        SELECT
            prediction_datetime,
            model_id,
            component,
            explain_type,
            feature_name,
            feature_value,
            effect_value,
            abs_effect_value,
            effect_rank,
            base_value
        FROM {PREDICTION_EXPLAIN_TABLE}
        WHERE area_name = :area_name
          AND prediction_model_id = :prediction_model_id
          AND prediction_datetime = :prediction_datetime
        ORDER BY
            model_id,
            component,
            effect_rank
    """),
    engine,
    params={
        "area_name": AREA_NAME,
        "prediction_model_id": PRED_MODEL_ID,
        "prediction_datetime": one_dt,
    }
)

print("prediction_datetime:", one_dt)
display(one_prediction_explain)